<div style="background-color:#FDD5DF; padding:10px; border-radius:14px; border-left:10px solid #FF8DA1;">
<h3><b> Mid-Term Assignment - CSV to Data Warehouse </b></h3>

Introduction
fazer um indice
explicar pq escoli este dataset(maybe so no video)
explicar a junção


In [1]:
import pandas as pd

In [2]:
df_1 = pd.read_csv('tmdb_5000_movies.csv')
df_2 = pd.read_csv('tmdb_5000_credits.csv')

In [3]:
# the 'id' and 'movie_id' are the same but with different names, so we combine based on that id
df_combined = pd.merge(df_1, df_2, left_on = 'id', right_on = 'movie_id')

In [ ]:
# the problem of this merge is that the df_combined will have both the 'id' and the 'movie_id', we need to remove one
df_combined.drop(columns= ['movie_id', 'title_y'], inplace = True)
df_combined.rename(columns= {'title_x' : 'title'})

**Tratar dos Missing Values**

In [5]:
df_combined.isna().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title_x                    0
vote_average               0
vote_count                 0
cast                       0
crew                       0
dtype: int64

In [6]:
df_combined = df_combined.dropna(subset=['release_date'])
df_combined = df_combined.dropna(subset=['runtime'])


In [7]:
df_combined.drop(columns = ['homepage', 'tagline', 'overview', 'keywords', 'status',], inplace = True)


In [8]:
df_combined.dtypes

budget                    int64
genres                   object
id                        int64
original_language        object
original_title           object
popularity              float64
production_companies     object
production_countries     object
release_date             object
revenue                   int64
runtime                 float64
spoken_languages         object
title_x                  object
vote_average            float64
vote_count                int64
cast                     object
crew                     object
dtype: object

In [9]:
df_combined['release_date'] = pd.to_datetime(df_combined['release_date'])
df_combined['budget'] = pd.to_numeric(df_combined['budget'], errors='coerce')
df_combined['revenue'] = pd.to_numeric(df_combined['revenue'], errors='coerce')

## **TRATAR DA COLUNA DOS GENRES**

In [11]:
fact_table = df_combined[['id','budget','revenue', 'runtime', 'popularity', 'vote_average','vote_count']]

In [12]:
dim1_date = df_combined[['id','release_date']]
dim1_date['year'] = df_combined['release_date'].dt.year.astype(int)
dim1_date['month'] = df_combined['release_date'].dt.month.astype(int)
dim1_date['quarter'] = df_combined['release_date'].dt.quarter.astype(int)



C:\Users\laura\AppData\Local\Temp\ipykernel_37448\3792899142.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dim1_date['year'] = df_combined['release_date'].dt.year.astype(int)
C:\Users\laura\AppData\Local\Temp\ipykernel_37448\3792899142.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dim1_date['month'] = df_combined['release_date'].dt.month.astype(int)
C:\Users\laura\AppData\Local\Temp\ipykernel_37448\3792899142.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [30]:
dim1_date

,id,release_date,year,month,quarter
0,19995,2009-12-10,2009,12,4
1,285,2007-05-19,2007,5,2
2,206647,2015-10-26,2015,10,4
3,49026,2012-07-16,2012,7,3
4,49529,2012-03-07,2012,3,1
...,...,...,...,...,...
4798,9367,1992-09-04,1992,9,3
4799,72766,2011-12-26,2011,12,4
4800,231617,2013-10-13,2013,10,4
4801,126186,2012-05-03,2012,5,2


In [16]:
import ast

In [17]:
df_combined['genres'] = df_combined['genres'].apply(ast.literal_eval)

In [22]:
df_genres_exploded = df_combined[['id', 'genres']].explode('genres')
df_genres_exploded = df_genres_exploded.dropna(subset=['genres'])
df_genres_exploded

,id,genres
0,19995,"{'id': 28, 'name': 'Action'}"
0,19995,"{'id': 12, 'name': 'Adventure'}"
0,19995,"{'id': 14, 'name': 'Fantasy'}"
0,19995,"{'id': 878, 'name': 'Science Fiction'}"
1,285,"{'id': 12, 'name': 'Adventure'}"
...,...,...
4797,231617,"{'id': 35, 'name': 'Comedy'}"
4797,231617,"{'id': 18, 'name': 'Drama'}"
4797,231617,"{'id': 10749, 'name': 'Romance'}"
4797,231617,"{'id': 10770, 'name': 'TV Movie'}"


In [24]:
dim_genres = pd.DataFrame({
    'movie_id': df_genres_exploded['id'],
    'genre_id': df_genres_exploded['genres'].apply(lambda x: x['id']),
    'genre_name': df_genres_exploded['genres'].apply(lambda x: x['name'])
})
dim_genres

,movie_id,genre_id,genre_name
0,19995,28,Action
0,19995,12,Adventure
0,19995,14,Fantasy
0,19995,878,Science Fiction
1,285,12,Adventure
...,...,...,...
4797,231617,35,Comedy
4797,231617,18,Drama
4797,231617,10749,Romance
4797,231617,10770,TV Movie
